<a href="https://colab.research.google.com/github/THDelak/LoRA/blob/main/Hands_On_Fine_Tuning_con_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass ***Fine-tuning y Evaluación de Modelos***, se proporciona el siguiente ***Colab*** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo Llama ligero, y medir su mejora con una métrica objetiva.

A diferencia de los Temas anteriores, aquí no usamos Groq — Groq solo sirve para inferencia, no para entrenar modelos. Usamos **Hugging Face** (librerías `transformers` y `peft`) directamente sobre la GPU gratuita de Colab.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1cKZ_hCf231RE84FDvGkEiKMH6ZDkVZzH?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura, añadiendo un nombre asociado al token para guardarlo dentro de una variable y usarlo dentro del notebook. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [19]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets

# Ejecutar en un runtime nuevo de Colab con GPU T4.
# Se conserva el torch con CUDA preinstalado en Colab (requiere torch >= 2.0).
%pip install -q transformers==4.48.3 peft==0.14.0 trl==0.15.2 datasets==3.2.0 accelerate==1.3.0 "huggingface-hub>=0.25,<1.0" "groq>=0.31,<1.0"

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTConfig, SFTTrainer
from importlib.metadata import version

# Crear el secreto HF_TOKEN y habilitar su acceso para este notebook.
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
for libreria in ("torch", "transformers", "peft", "trl", "datasets", "accelerate"):
    print(f"{libreria}: {version(libreria)}")


torch: 2.11.0+cu128
transformers: 4.48.3
peft: 0.14.0
trl: 0.15.2
datasets: 3.2.0
accelerate: 1.3.0


### **CARGAR EL MODELO BASE**

Usamos un modelo Llama ligero (pocos parámetros) para que el fine-tuning corra en minutos sobre la GPU T4 gratuita de Colab, sin necesitar cuantización adicional.

In [20]:
# Cargar el modelo base de Llama y su tokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Activa GPU en Colab: Entorno de ejecucion > Cambiar tipo de entorno de ejecucion.")

set_seed(42)
modelo_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(modelo_id)
# Usar UNK para padding conserva EOS como objetivo de aprendizaje.
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = "right"
modelo = AutoModelForCausalLM.from_pretrained(
    modelo_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
).to("cuda")
modelo.config.pad_token_id = tokenizer.pad_token_id


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [21]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo

def generar_respuesta(prompt, max_new_tokens=100):
    mensajes = [{"role": "user", "content": prompt}]
    texto = tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )
    entradas = tokenizer(texto, return_tensors="pt", add_special_tokens=False).to(modelo.device)
    estaba_entrenando = modelo.training
    modelo.eval()
    try:
        with torch.inference_mode():
            salida = modelo.generate(
                **entradas,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
    finally:
        modelo.train(estaba_entrenando)
    return tokenizer.decode(
        salida[0, entradas["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

prompt_prueba = "Que es el fine-tuning?"
respuesta_base = generar_respuesta(prompt_prueba)
print("Modelo base:", respuesta_base)


Modelo base: El fine-tuning es una técnica de aprendizaje automático que se utiliza para mejorar el modelo previamente entrenado. En este proceso, el modelo se entrena con datos adicionales, que se agregan a los datos que se entrenaron para mejorar su capacidad de predicción. El objetivo es mejorar la precisión del modelo en nuevos datos, lo que se hace mediante la incorporación de nuevos


### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que muestren el comportamiento que queremos enseñarle al modelo. Con pocos ejemplos (5 a 10) es suficiente para una demo — no es un dataset de producción.

In [22]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset

# Cinco ejemplos didacticos: respuestas breves con un formato consistente.
ejemplos = [
    {"entrada": "Que es el fine-tuning?",
     "respuesta": "En breve: el fine-tuning adapta un modelo preentrenado con ejemplos de una tarea concreta."},
    {"entrada": "Que es LoRA?",
     "respuesta": "En breve: LoRA entrena matrices pequenas adicionales y mantiene congelados los pesos originales."},
    {"entrada": "Que es un tokenizer?",
     "respuesta": "En breve: un tokenizer convierte texto en tokens que el modelo puede procesar."},
    {"entrada": "Que es la perdida de entrenamiento?",
     "respuesta": "En breve: la perdida mide el error al predecir los tokens de los ejemplos de entrenamiento."},
    {"entrada": "Para que sirve un dataset?",
     "respuesta": "En breve: un dataset aporta ejemplos del comportamiento que queremos ensenar al modelo."},
]
dataset = Dataset.from_list(ejemplos)

def formatear_ejemplo(ejemplo):
    mensajes = [
        {"role": "user", "content": ejemplo["entrada"]},
        {"role": "assistant", "content": ejemplo["respuesta"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=False
    )}

dataset = dataset.map(formatear_ejemplo, remove_columns=dataset.column_names)
# Tokenizar sin duplicar los tokens especiales de la plantilla de chat.
dataset = dataset.map(lambda lote: tokenizer(
    lote["text"], add_special_tokens=False, truncation=True, max_length=256
), batched=True, remove_columns=["text"])


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [23]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base

set_seed(42)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
modelo = get_peft_model(modelo, lora_config)
modelo.print_trainable_parameters()
modelo.config.use_cache = False


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [24]:
# Configurar el entrenador (SFTTrainer) y ejecutar el fine-tuning

training_args = SFTConfig(
    output_dir="/content/tinyllama-lora",
    num_train_epochs=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=1,
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    max_seq_length=256,
    packing=False,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42,
)
trainer = SFTTrainer(
    model=modelo,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset,
    processing_class=tokenizer,
)
# Evaluar los mismos ejemplos antes y despues: mide ajuste, no generalizacion.
# Con la inicializacion predeterminada, LoRA no altera la salida inicial.
perdida_inicial = trainer.evaluate()["eval_loss"]
resultado_entrenamiento = trainer.train()
perdida_final = trainer.evaluate()["eval_loss"]


Converting train dataset to ChatML:   0%|          | 0/5 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/5 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Step,Training Loss
1,2.470400
2,2.734600
3,2.523400
4,2.131500
5,3.049000
6,2.252900
7,2.912500
8,2.468900
9,2.079500
10,1.809300


### **DESPUÉS DEL FINE-TUNING: MEDIR LA MEJORA**

Compararemos la pérdida antes y después del entrenamiento como métrica objetiva.

In [25]:
# Comparar la pérdidas

print(f"Perdida antes del entrenamiento: {perdida_inicial:.4f}")
print(f"Perdida despues del entrenamiento: {perdida_final:.4f}")
reduccion = perdida_inicial - perdida_final
reduccion_porcentual = 100 * reduccion / perdida_inicial if perdida_inicial else float("nan")
print(f"Reduccion absoluta: {reduccion:.4f} ({reduccion_porcentual:.2f}%)")
perdidas_entrenamiento = [
    {"paso": registro["step"], "loss": registro["loss"]}
    for registro in trainer.state.log_history if "loss" in registro
]
print("Historial de perdida de entrenamiento:", perdidas_entrenamiento)
print("Evaluacion sobre los ejemplos de entrenamiento; no mide generalizacion.")


Perdida antes del entrenamiento: 2.6773
Perdida despues del entrenamiento: 0.0414
Reduccion absoluta: 2.6359 (98.45%)
Historial de perdida de entrenamiento: [{'paso': 1, 'loss': 2.4704}, {'paso': 2, 'loss': 2.7346}, {'paso': 3, 'loss': 2.5234}, {'paso': 4, 'loss': 2.1315}, {'paso': 5, 'loss': 3.049}, {'paso': 6, 'loss': 2.2529}, {'paso': 7, 'loss': 2.9125}, {'paso': 8, 'loss': 2.4689}, {'paso': 9, 'loss': 2.0795}, {'paso': 10, 'loss': 1.8093}, {'paso': 11, 'loss': 1.9248}, {'paso': 12, 'loss': 2.1866}, {'paso': 13, 'loss': 1.8562}, {'paso': 14, 'loss': 2.4595}, {'paso': 15, 'loss': 1.5434}, {'paso': 16, 'loss': 1.4858}, {'paso': 17, 'loss': 1.6557}, {'paso': 18, 'loss': 2.1674}, {'paso': 19, 'loss': 1.3355}, {'paso': 20, 'loss': 1.653}, {'paso': 21, 'loss': 1.408}, {'paso': 22, 'loss': 0.9884}, {'paso': 23, 'loss': 1.4627}, {'paso': 24, 'loss': 1.1136}, {'paso': 25, 'loss': 1.8467}, {'paso': 26, 'loss': 1.0284}, {'paso': 27, 'loss': 1.0981}, {'paso': 28, 'loss': 0.6612}, {'paso': 29, '

In [26]:
# Referencia cualitativa vs Referencia con un modelo de producción

respuesta_ajustada = generar_respuesta(prompt_prueba)
print("Prompt:", prompt_prueba)
print("TinyLlama antes:", respuesta_base)
print("TinyLlama con LoRA:", respuesta_ajustada)

# Comparacion externa indicada en el notebook: requiere GROQ_API_KEY en Secrets.
# Es opcional para el entrenamiento y no interviene en el calculo de la perdida.
from groq import Groq
try:
    groq_api_key = userdata.get("GROQ_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    groq_api_key = None

if groq_api_key:
    cliente_groq = Groq(api_key=groq_api_key)
    respuesta_groq = cliente_groq.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt_prueba}],
        temperature=0,
        max_completion_tokens=1024,
    ).choices[0].message.content
    print("Groq (GPT-OSS-20B):", respuesta_groq)
else:
    print("Comparacion con Groq pendiente: configura el secreto GROQ_API_KEY.")


Prompt: Que es el fine-tuning?
TinyLlama antes: El fine-tuning es una técnica de aprendizaje automático que se utiliza para mejorar el modelo previamente entrenado. En este proceso, el modelo se entrena con datos adicionales, que se agregan a los datos que se entrenaron para mejorar su capacidad de predicción. El objetivo es mejorar la precisión del modelo en nuevos datos, lo que se hace mediante la incorporación de nuevos
TinyLlama con LoRA: En breve: el fine-tuning adapta un modelo preentrenado con ejemplos de una tarea concreta.
Groq (GPT-OSS-20B): **Fine‑tuning** (ajuste fino) es una técnica de *transfer learning* que permite adaptar un modelo previamente entrenado (pre‑entrenado) a una tarea específica con un conjunto de datos más pequeño y especializado. En lugar de entrenar el modelo desde cero, se parte de los pesos ya aprendidos y se “ajustan” ligeramente para que el modelo se desempeñe mejor en la nueva tarea.

---

## 1. ¿Por qué usar fine‑tuning?

| Ventaja | Explicación |


**Nota:** Por eso, además de la respuesta de TinyLlama, se muestra una respuesta de Groq (GPT-OSS-20B) para el mismo prompt: no porque el proceso de fine-tuning haya fallado, sino como punto de comparación de cómo respondería un modelo de producción con muchos más parámetros y datos de entrenamiento — TinyLlama con 5 ejemplos demuestra la técnica de LoRA, no busca igualar la calidad de un modelo así de grande.

## **CHALLENGE: AJUSTE DE TONO CON LORA**

Una vez visto el ***Hands-On: Fine-tuning con LoRA***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se ajustará un modelo Llama ligero con un dataset propio para enseñarle un tono o formato de respuesta específico, comparando la pérdida **antes** y **después** del fine-tuning como métrica objetiva. En esta solución se usa como ejemplo un asistente de dudas frecuentes del propio curso.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Carga el modelo y define tu dataset:**

   * Instala las librerías, inicia sesión en Hugging Face con tu token y carga el modelo base junto con la función `generar_respuesta`.

   * Construye una lista llamada `ejemplos` con al menos 4 pares de entrada/respuesta que reflejen el tono o formato que quieres enseñarle al modelo, y conviértela en `dataset`.

In [27]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets

# Reutilizar las dependencias y la sesion de Hugging Face de la celda 6.
# Ejecutar primero esa celda si se inicia directamente desde el Challenge.
import gc
from importlib.metadata import version
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTConfig, SFTTrainer

versiones_esperadas = {
    "transformers": "4.48.3", "peft": "0.14.0", "trl": "0.15.2",
    "datasets": "3.2.0", "accelerate": "1.3.0",
}
for libreria, esperada in versiones_esperadas.items():
    if version(libreria) != esperada:
        raise RuntimeError(f"Ejecuta la celda 6 en un runtime nuevo: se requiere {libreria}=={esperada}.")


In [28]:
# Cargar el modelo base de Llama y su tokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Activa una GPU T4 en el entorno de ejecucion de Colab.")

# El Hands-On ya entreno adaptadores: cargar pesos base limpios evita contaminacion.
# Liberar tambien el entrenador, que conserva referencias al modelo y optimizador.
for nombre in ("trainer", "modelo"):
    globals().pop(nombre, None)
gc.collect()
torch.cuda.empty_cache()
set_seed(42)
modelo_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# from_pretrained reutiliza los archivos descargados en la cache de Hugging Face.
tokenizer = AutoTokenizer.from_pretrained(modelo_id)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = "right"
modelo = AutoModelForCausalLM.from_pretrained(
    modelo_id, torch_dtype=torch.float16, use_safetensors=True,
).to("cuda")
modelo.config.pad_token_id = tokenizer.pad_token_id


In [29]:
# Definir la funcion de generacion de texto

def generar_respuesta(prompt, max_new_tokens=100):
    mensajes = [{"role": "user", "content": prompt}]
    texto = tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )
    entradas = tokenizer(texto, return_tensors="pt", add_special_tokens=False).to(modelo.device)
    estaba_entrenando = modelo.training
    modelo.eval()
    try:
        with torch.inference_mode():
            salida = modelo.generate(
                **entradas,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
    finally:
        modelo.train(estaba_entrenando)
    return tokenizer.decode(
        salida[0, entradas["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


In [30]:
# Definir la lista ejemplos y convertirla en dataset

# Dataset propio: diagnosticos sencillos con tres lineas y tono tecnico breve.
ejemplos = [
    {"entrada": "Mi cliente no conecta con la API y muestra Connection refused.",
     "respuesta": "Problema: Conexion a la API rechazada.\nCausa: El servicio no escucha en el puerto indicado.\nSolución: Comprueba la URL y el puerto e inicia el servicio."},
    {"entrada": "La aplicacion indica que falta la variable de entorno APP_MODE.",
     "respuesta": "Problema: APP_MODE no esta definida.\nCausa: La variable no se cargo en el proceso.\nSolución: Define APP_MODE en el entorno y reinicia la aplicacion."},
    {"entrada": "Solicito /clientes y el servidor devuelve HTTP 404.",
     "respuesta": "Problema: La ruta /clientes no se encuentra.\nCausa: La URL no coincide con una ruta publicada.\nSolución: Revisa la ruta y el prefijo de la API."},
    {"entrada": "La aplicacion pierde la conexion con PostgreSQL al detenerse el servidor.",
     "respuesta": "Problema: PostgreSQL no esta disponible.\nCausa: El servidor de base de datos esta detenido.\nSolución: Inicia PostgreSQL y comprueba la conexion desde la aplicacion."},
    {"entrada": "La API devuelve HTTP 401 cuando envio un token vencido.",
     "respuesta": "Problema: La API rechaza la autenticacion.\nCausa: El token de acceso ha vencido.\nSolución: Renueva el token y actualiza la cabecera Authorization."},
    {"entrada": "Python muestra ModuleNotFoundError: No module named requests.",
     "respuesta": "Problema: Python no encuentra requests.\nCausa: La dependencia falta en el entorno activo.\nSolución: Ejecuta python -m pip install requests en ese entorno."},
    {"entrada": "Mi servidor no inicia porque el puerto 8000 ya esta ocupado.",
     "respuesta": "Problema: El puerto 8000 esta ocupado.\nCausa: Otro proceso utiliza ese puerto.\nSolución: Identifica el proceso y detenlo si corresponde, o configura otro puerto."},
]
dataset = Dataset.from_list(ejemplos)

def formatear_ejemplo(ejemplo):
    mensajes = [
        {"role": "user", "content": ejemplo["entrada"]},
        {"role": "assistant", "content": ejemplo["respuesta"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=False
    )}

dataset = dataset.map(formatear_ejemplo, remove_columns=dataset.column_names)
dataset = dataset.map(lambda lote: tokenizer(
    lote["text"], add_special_tokens=False, truncation=True, max_length=256
), batched=True, remove_columns=["text"])


Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

**2. Prueba el modelo base:** Genera una respuesta con el modelo sin ajustar para un prompt de prueba y guárdala en `respuesta_base`.

In [31]:
# Probar el modelo base con un prompt de prueba y guardar el resultado en respuesta_base

prompt_prueba = "Al arrancar mi servicio en el puerto 3000 aparece Address already in use. Que debo revisar?"
assert prompt_prueba not in [ejemplo["entrada"] for ejemplo in ejemplos]
# Guardar el string antes de aplicar LoRA; no volver a calcularlo tras entrenar.
respuesta_base = generar_respuesta(prompt_prueba, max_new_tokens=160)
print("Modelo base:", respuesta_base)


Modelo base: Sí, es posible que el servicio que deseas iniciar en el puerto 3000 ya está siendo utilizado por otro servicio. Para resolver este error, debes revisar si el puerto 3000 está siendo utilizado por otro servicio.

Para revisar si el puerto 3000 está siendo utilizado por otro servicio, puedes utilizar el comando `netstat -anp` en el terminal. Este comando mostrará la lista de puertos que están siendo utilizados por los servicios en ejecución.

Para verificar si el puerto 3000 está siendo utilizado por otro servicio, busca


**3. Configura y aplica LoRA:** Fija una semilla con `set_seed` y define tu `LoraConfig` (rango, alpha, módulos objetivo, dropout en 0) y aplícalo al modelo base.

In [32]:
# Configurar LoraConfig y aplicarlo al modelo base

set_seed(42)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
modelo = get_peft_model(modelo, lora_config)
modelo.print_trainable_parameters()
modelo.config.use_cache = False


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


**4. Entrena:** Configura el `SFTTrainer` con tu dataset y ejecuta el fine-tuning; guarda la pérdida final en `perdida_final`.

In [33]:
# Configurar el Trainer con el dataset y ejecutar el fine-tuning; guardar la pérdida final en perdida_final

training_args = SFTConfig(
    output_dir="/content/tinyllama-lora-challenge",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=1,
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    max_seq_length=256,
    packing=False,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42,
)
trainer = SFTTrainer(
    model=modelo,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset,
    processing_class=tokenizer,
)
# Evaluar los mismos ejemplos antes y despues: mide ajuste, no generalizacion.
# Con la inicializacion predeterminada, LoRA no altera la salida inicial.
perdida_inicial = trainer.evaluate()["eval_loss"]
resultado_entrenamiento = trainer.train()
perdida_final = trainer.evaluate()["eval_loss"]


Converting train dataset to ChatML:   0%|          | 0/7 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/7 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Step,Training Loss
1,2.140400
2,1.686100
3,1.767400
4,1.871800
5,1.776000
6,2.032000
7,1.875300
8,1.557100
9,1.863400
10,1.727900


**5. Compara y concluye:** Calcula la reducción entre la pérdida inicial y `perdida_final` como métrica objetiva, y usa la respuesta generada por el modelo ya ajustado solo como referencia cualitativa.

In [34]:
# Comparar la perdida inicial y final del entrenamiento como metrica objetiva

reduccion_loss = perdida_inicial - perdida_final
porcentaje_reduccion = (100 * reduccion_loss / perdida_inicial) if perdida_inicial > 0 else None
print(f"Perdida inicial: {perdida_inicial:.4f}")
print(f"Perdida final: {perdida_final:.4f}")
print(f"Reduccion absoluta: {reduccion_loss:.4f}")
if porcentaje_reduccion is not None:
    print(f"Reduccion porcentual: {porcentaje_reduccion:.2f} %")
else:
    print("Reduccion porcentual: no definida para una perdida inicial nula.")
print("Estas perdidas se evaluan sobre el dataset de entrenamiento; no miden generalizacion.")
if reduccion_loss > 0:
    print("La perdida disminuyo: el modelo se ajusto mejor a estos ejemplos.")
else:
    print("La perdida no disminuyo en esta ejecucion.")


Perdida inicial: 1.9878
Perdida final: 0.1476
Reduccion absoluta: 1.8403
Reduccion porcentual: 92.58 %
Estas perdidas se evaluan sobre el dataset de entrenamiento; no miden generalizacion.
La perdida disminuyo: el modelo se ajusto mejor a estos ejemplos.


In [35]:
# Como referencia cualitativa (puede variar de sesión a sesión con un dataset tan chico):

respuesta_ajustada = generar_respuesta(prompt_prueba, max_new_tokens=160)
print("Pregunta:", prompt_prueba)
print("\n=== ANTES DEL FINE-TUNING ===")
print(respuesta_base)
print("\n=== DESPUÉS DEL FINE-TUNING ===")
print(respuesta_ajustada)
print("\nRevisa si aparecen Problema:, Causa: y Solución: en tres lineas breves.")
print("La adopcion del formato se observa en las respuestas reales; no se garantiza con siete ejemplos.")


Pregunta: Al arrancar mi servicio en el puerto 3000 aparece Address already in use. Que debo revisar?

=== ANTES DEL FINE-TUNING ===
Sí, es posible que el servicio que deseas iniciar en el puerto 3000 ya está siendo utilizado por otro servicio. Para resolver este error, debes revisar si el puerto 3000 está siendo utilizado por otro servicio.

Para revisar si el puerto 3000 está siendo utilizado por otro servicio, puedes utilizar el comando `netstat -anp` en el terminal. Este comando mostrará la lista de puertos que están siendo utilizados por los servicios en ejecución.

Para verificar si el puerto 3000 está siendo utilizado por otro servicio, busca

=== DESPUÉS DEL FINE-TUNING ===
Problema: Address already in use.
Causa: La puerta de acceso 3000 esta ocupada.
Solución: Vea las rutinas y servicios que utilizan esta puerta de acceso y desactiva o modifica las direcciones.

Revisa si aparecen Problema:, Causa: y Solución: en tres lineas breves.
La adopcion del formato se observa en las r